# Dataset

In [79]:
text = ""
with open("tinyshakesphere.txt") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [80]:
vocab = list(set(text))
vocab_size = len(vocab)
print(vocab_size)

65


In [81]:
encode = {val:i for i,val in enumerate(vocab)}
decode = {i:val for i,val in enumerate(vocab)}

print(decode[0])

r


In [82]:
import torch
tokenized_text = torch.tensor([encode[c] for c in text],dtype=torch.long)
print(tokenized_text[:1000])

tensor([30,  7,  0, 33, 39, 45, 58,  7, 39,  7, 62, 42, 14, 37,  8, 27, 42, 63,
        21,  0, 42, 45,  4, 42, 45, 55,  0, 21, 54, 42, 42, 26, 45, 28, 14, 57,
        45, 63, 31,  0, 39, 46, 42,  0, 38, 45, 46, 42, 28,  0, 45, 47, 42, 45,
        33, 55, 42, 28, 16,  5,  8,  8,  1, 53, 53, 37,  8, 13, 55, 42, 28, 16,
        38, 45, 33, 55, 42, 28, 16,  5,  8,  8, 30,  7,  0, 33, 39, 45, 58,  7,
        39,  7, 62, 42, 14, 37,  8, 32, 21, 31, 45, 28,  0, 42, 45, 28, 53, 53,
        45,  0, 42, 33, 21, 53, 19, 42, 26, 45,  0, 28, 39, 46, 42,  0, 45, 39,
        21, 45, 26,  7, 42, 45, 39, 46, 28, 14, 45, 39, 21, 45, 63, 28, 47,  7,
        33, 46, 43,  8,  8,  1, 53, 53, 37,  8, 64, 42, 33, 21, 53, 19, 42, 26,
         5, 45,  0, 42, 33, 21, 53, 19, 42, 26,  5,  8,  8, 30,  7,  0, 33, 39,
        45, 58,  7, 39,  7, 62, 42, 14, 37,  8, 30,  7,  0, 33, 39, 38, 45, 57,
        21, 31, 45, 16, 14, 21,  4, 45, 58, 28,  7, 31, 33, 45, 34, 28,  0, 54,
         7, 31, 33, 45,  7, 33, 45, 54, 

In [83]:
train_data = tokenized_text[:int(0.9*len(text))]
val_data = tokenized_text[int(0.9*len(text)):]

print(len(train_data),len(val_data))

1003854 111540


In [84]:
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data

    random_starting_indices = torch.randint(len(data) - block_size, (block_size,))

    x = torch.stack([data[i:i+block_size] for i in random_starting_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_starting_indices])
    return x,y
    

x,y = get_batch("train")
print(x)
print(y)


tensor([[21, 53, 16, 33, 38,  8, 46, 21],
        [ 3,  6, 41, 11, 32, 58,  3, 13],
        [45, 26,  7, 19, 42,  0, 33, 45],
        [ 0, 45, 33, 28, 54,  0, 42, 26],
        [45, 29,  0,  7, 14, 50, 45, 47],
        [28, 39,  7, 21, 14, 45, 63, 21],
        [ 8,  1, 14, 26, 45, 29,  0, 42],
        [33,  4, 42, 28,  0, 45, 39, 21]])
tensor([[53, 16, 33, 38,  8, 46, 21,  4],
        [ 6, 41, 11, 32, 58,  3, 13, 37],
        [26,  7, 19, 42,  0, 33, 45, 26],
        [45, 33, 28, 54,  0, 42, 26, 45],
        [29,  0,  7, 14, 50, 45, 47, 42],
        [39,  7, 21, 14, 45, 63, 21,  0],
        [ 1, 14, 26, 45, 29,  0, 42, 28],
        [ 4, 42, 28,  0, 45, 39, 21,  8]])


# Bigram Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BigramModel(nn.Module):
    def __init__(self,n_embeddings, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=n_embeddings, embedding_dim=vocab_size)

    def forward(self,input_encodings,targets_encodings=None):
        input_embeddings = self.embedding(input_encodings) # (B,T,C) , B = block_size, T = length of sequence, C = embedding_dim

        loss = None
        if targets_encodings is not None:
            input_embeddings_for_loss = input_embeddings.permute(0,2,1) # (B,C,T)
            loss = F.cross_entropy(input_embeddings_for_loss,targets_encodings)
        
        return loss, input_embeddings 
    
    def generate(self,input_encoding, max_output_len):
        for _ in range(max_output_len):
            _,logits = self(input_encoding)
            logits = logits[:,-1,:] # Last char of sequence
            predictions_probs = F.softmax(logits,dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding,prediction),dim=1)

        return input_encoding
    
model = BigramModel(vocab_size, vocab_size) # 1 row for each possible char, and for softmax, we need the vocab as output

start_context = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))



tensor([[ 0, 57, 13, 30, 43, 61, 60,  5,  7, 41, 53, 32,  3, 54, 49,  4, 37, 43,
         36,  9, 18, 31, 46, 38, 22, 35, 58, 12, 28,  0,  3, 16,  1, 43,  9, 48,
         38, 36,  0, 36, 60, 30, 56, 10, 34, 13, 40, 44, 16, 35, 21, 11, 36,  8,
         18, 31, 17, 16, 37, 43, 15, 45, 54, 48,  0, 54,  1, 18, 18, 38, 62, 64,
         57, 23, 35, 47, 62, 15, 28,  8, 26, 59, 37, 60,  9,  6, 23, 35, 38, 49,
         17, 61, 48, 58, 53,  3, 42, 31, 22, 36, 52]])
rySF?xj.iOlYUcPw:?'3$uh,Q&C;arUkA?3N,'r'jFqWMSGKk&oL'
$uDk:?E cNrcA$$,zRyV&mzEa
d!:j3TV&,PDxNClUeuQ'-
